# Clean re-run of the sonar detection experiments

I am retraining all four models on the corrected pipeline and scoring them against one ground
truth, one split, and one postprocessing config.

My original comparison could not support its conclusion because three implementation defects each
favoured DCCAN over the baselines I was comparing it against, and the splits leaked 219 of 242
validation images into the labelled training set. I wrote all of that up in `docs/AUDIT.md`. This
notebook is how I re-run the experiment properly.

Before running anything: Runtime, then Change runtime type, then pick A100 or L4.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('CUDA available:', torch.cuda.is_available())


## 1. Mount Drive and clone the repo

Checkpoints go to Drive because Colab wipes `/content` when a session drops, and DCCAN takes
about two hours. The dataset is committed inside the repo, so the clone brings it with it and I
have nothing to upload.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')

!git clone -q https://github.com/Kablan-ASBN/sonar-object-detection.git /content/sonar
%cd /content/sonar
!pip install -q -e '.[dev]'

import os
os.makedirs('/content/drive/MyDrive/sonar-runs', exist_ok=True)
os.makedirs('/content/sonar/preds', exist_ok=True)

!du -sh data && git log -1 --oneline


## 2. Check the splits before I train anything

This has to exit 0. It is the check that would have caught my original leak, and it runs in CI on
every push. If it reports leakage I stop here, because nothing downstream would mean anything.


In [ ]:
!sonar audit leakage --root raw=data/line2voc \
                     --root denoised=data/line2voc_preprocessed \
                     --root augmented=data/line2voc_preprocessed_augmented


## 3. Train the four models

I run these one cell at a time. If the session drops I lose one model, not four. On an A100 the
baselines take roughly 40 minutes each, DANN about 1.5 hours, and DCCAN about 2 hours.

`--mode raw` is not optional. It writes a sidecar recording that no score floor was applied, and
`sonar eval` refuses a filtered file unless I pass `--allow-filtered`. I never pass that flag for a
number I intend to report. Mixing a thresholded export with an unthresholded one is exactly the
defect that made my original baselines look worse than they were.


### `baseline_raw`

Trained on raw sonar, the only model in my original comparison that was not affected by the leak.


In [ ]:
CFG = "baseline_raw"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true


### `baseline_denoised`

Trained on the median filtered source domain.


In [ ]:
CFG = "baseline_denoised"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true


### `dann`

Global adversarial alignment, now with the domain features going through `model.transform`.


In [ ]:
CFG = "dann"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true


### `dccan`

The three path architecture i proposed, with the proxy classifier actually trained.


In [ ]:
CFG = "dccan"
CKPT = f"/content/drive/MyDrive/sonar-runs/{CFG}.pt"
CSV  = f"/content/sonar/preds/{CFG}.csv"

!sonar train --config configs/{CFG}.yaml --out {CKPT} --device cuda

!sonar predict --checkpoint {CKPT} --root data/line2voc --split test \
               --out {CSV} --mode raw --device cuda

!cp {CSV} {CSV}.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null || true


## 4. Score all four against the same ground truth

One `--gt-root`, one `--split`, four models, one postprocessing config. This is the comparison my
original one was not: back then CLAHE+Aug was scored against a different dataset root whose
validation split shared only 23 of its ids with the one the other four models used.

`GroundTruth` in this codebase is bound to a single root and a single split, and `compare` takes
exactly one, so mixing them is now a type error rather than something I have to remember.


In [ ]:
!sonar eval --gt-root data/line2voc --split test \
  --preds raw=preds/baseline_raw.csv \
  --preds denoised=preds/baseline_denoised.csv \
  --preds dann=preds/dann.csv \
  --preds dccan=preds/dccan.csv \
  --froc /content/drive/MyDrive/sonar-runs/froc.csv


## 5. Save everything to Drive

So the predictions and the FROC curve survive the session.


In [ ]:
!cp -v preds/*.csv preds/*.meta.json /content/drive/MyDrive/sonar-runs/ 2>/dev/null | tail -5
!ls -la /content/drive/MyDrive/sonar-runs/


## How I plan to read the result

Four corrections all move against DCCAN compared to my original run:

1. the baselines are no longer score floored, so they keep the low confidence tail that COCO AP
   integrates over
2. the baselines and DANN no longer train on mirrored images with unmirrored boxes
3. DANN's discriminator now sees the same normalised, resized input the detector sees
4. nothing is evaluated on images it was trained on

My original margin was 0.011 AP50, and each of those corrections is larger than that. So I expect
the gap to shrink and I would not be surprised if it disappears. If the baselines come out ahead,
that is the result and I will report it. I am not going to tune until DCCAN wins, because that is
how the first result happened.

The one claim from the original work I still stand behind either way: the three path design trains
stably under mixed precision, where a standalone CDAN outer product mapping did not.
